# Lab 6: Path Tracking and Control — PID & Pure Pursuit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatayer8891-boop/zuj-robotics/blob/main/labs/lab-06-tracking-control/notebook.ipynb)

---

**Course:** Introduction to Robotics for AI and Data Science (0135343)  
**Instructor:** Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/)  
**University:** Al-Zaytoonah University of Jordan

---

## Overview

Implement PID and Pure Pursuit controllers to make the robot smoothly follow planned paths.

> **80/20 Principle:** Focus on understanding the core algorithm in each activity. The implementation details will follow naturally once you grasp the concept.


In [ ]:
# ============================================================
# COLAB ENVIRONMENT SETUP
# ============================================================
# This cell installs the required packages for running this lab
# in Google Colab. If you're running locally, you can skip this.
# ============================================================

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Setting up Colab environment...")
    !pip install pybullet numpy matplotlib opencv-python-headless -q
    print("✅ All packages installed!")
    print("⚠️  Note: PyBullet runs in HEADLESS mode on Colab (no 3D GUI).")
    print("   You'll see matplotlib plots instead of the live 3D window.")
    print("   For the full 3D experience, run locally with: conda activate robotics_YourName")
else:
    print("✅ Running locally — full GUI mode available.")


---

## Activity 1: Activity1 Pid

**File:** `activity1_pid.py`

Run the code below to complete this activity.


In [ ]:
"""
Lab 6: Path Tracking
Activity 1: PID Path Follower
Author: Dr. Akram Fatayer

Description:
    A unicycle robot follows a sine-wave path using a PID controller acting on
    the Cross-Track Error (CTE). Your task is to tune the gains to achieve a
    critically-damped response.

NOTE ON SCOPE (see lecture Slide 3):
    The lecture explains that a path tracker must drive BOTH the Cross-Track
    Error (CTE) and the Heading Error (psi) to zero. This activity uses a
    simplified PID that corrects CTE only -- it is the universal baseline.
    The Stanley controller (lecture Slide 9) is the one that explicitly
    combines CTE and heading error; we compare against Pure Pursuit in
    Activity 2.
"""

import numpy as np
import matplotlib.pyplot as plt

# --- ROBOT PARAMETERS ---
DT = 0.1          # Time step (seconds)
V  = 2.0          # Constant forward velocity (m/s)
L  = 2.5          # Wheelbase of the robot (meters)

# --- PID GAINS (TUNE THESE) ---
# TODO: Tune these gains to achieve smooth, critically-damped path following.
# Start from the presets in PRESETS below and observe the three damping regimes.
Kp = 0.8   # Proportional gain
Ki = 0.0   # Integral gain
Kd = 2.0   # Derivative gain

# Preset gain sets that demonstrate the three damping regimes from lecture
# Slide 6. Try each by copying its values into Kp/Ki/Kd above, or run
# `python activity1_pid.py --compare` to see all three at once.
PRESETS = {
    "underdamped":       dict(Kp=2.0,  Ki=0.0, Kd=0.0),  # oscillates / weaves
    "overdamped":        dict(Kp=0.05, Ki=0.0, Kd=0.5),  # sluggish, lags / cuts corners
    "critically_damped": dict(Kp=0.8,  Ki=0.0, Kd=2.0),  # smooth, ideal
}


def generate_path():
    """Generates a sine wave path for the robot to follow."""
    x = np.linspace(0, 50, 500)
    y = 5 * np.sin(x / 5.0)
    return np.column_stack((x, y))


def calculate_cte(robot_pos, path):
    """
    Calculates the signed Cross-Track Error (CTE).

    Magnitude = distance from the robot to the nearest path point.
    Sign tells us which side of the path the robot is on:
        cross(path_tangent, robot_offset) > 0  ->  robot LEFT of path  -> CTE negative
        cross(path_tangent, robot_offset) < 0  ->  robot RIGHT of path -> CTE positive

    This sign convention matters: the control law below is written to match it.
    """
    distances   = np.linalg.norm(path - robot_pos, axis=1)
    nearest_idx = np.argmin(distances)
    cte         = distances[nearest_idx]

    if nearest_idx < len(path) - 1:
        path_vec = path[nearest_idx + 1] - path[nearest_idx]
    else:
        path_vec = path[nearest_idx] - path[nearest_idx - 1]

    robot_vec  = robot_pos - path[nearest_idx]
    # 2-D scalar cross product (avoids NumPy 2.0 deprecation of np.cross on 2-vectors)
    cross_prod = path_vec[0] * robot_vec[1] - path_vec[1] * robot_vec[0]

    if cross_prod > 0:
        cte = -cte    # robot is to the LEFT -> negative CTE

    return cte


def run_simulation(Kp, Ki, Kd, steps=400):
    """
    Simulates the robot following the path with the given PID gains.
    Returns (path, history_x, history_y, history_cte).
    """
    path = generate_path()

    # Initial state [x, y, theta] -- start off the path so there is an error.
    robot_state = np.array([0.0, 2.0, 0.0])

    int_cte  = 0.0
    prev_cte = 0.0

    history_x   = [robot_state[0]]
    history_y   = [robot_state[1]]
    history_cte = []

    for _ in range(steps):
        robot_pos = robot_state[:2]
        theta     = robot_state[2]

        # 1. Error
        cte = calculate_cte(robot_pos, path)
        history_cte.append(cte)

        # 2. PID control law
        #
        #   u(t) = Kp*e(t) + Ki * INT e(t) dt + Kd * de(t)/dt     (lecture Slide 5)
        #
        # u is the steering angle (delta). With the CTE sign convention in
        # calculate_cte() (negative when the robot is LEFT of the path), a
        # POSITIVE steering angle turns the robot left -- so the terms are
        # added with a POSITIVE sign to steer back toward the path.
        int_cte += cte * DT
        diff_cte = (cte - prev_cte) / DT
        steer    = Kp * cte + Ki * int_cte + Kd * diff_cte

        # Physical steering limit (+/- 30 degrees)
        max_steer = np.radians(30)
        steer     = np.clip(steer, -max_steer, max_steer)

        prev_cte = cte

        # 3. Unicycle kinematics update
        robot_state[0] += V * np.cos(theta) * DT
        robot_state[1] += V * np.sin(theta) * DT
        robot_state[2] += (V / L) * np.tan(steer) * DT

        history_x.append(robot_state[0])
        history_y.append(robot_state[1])

        if robot_state[0] >= path[-1, 0]:
            break

    return path, history_x, history_y, history_cte


def plot_single(Kp, Ki, Kd):
    """Run and plot a single PID configuration."""
    print(f"Running PID simulation (Kp={Kp}, Ki={Ki}, Kd={Kd}) ...")
    path, hx, hy, hcte = run_simulation(Kp, Ki, Kd)
    print("Simulation complete.")

    steady = [abs(c) for c in hcte[50:]] or [abs(c) for c in hcte]
    print(f"  Steady-state |CTE|: max={max(steady):.3f} m, "
          f"mean={np.mean(steady):.3f} m")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot(path[:, 0], path[:, 1], 'k--', linewidth=1.5, label='Reference path')
    ax1.plot(hx, hy, '-', color='#2980B9', linewidth=2, label='Robot trajectory')
    ax1.set_title(f'PID Path Following\n(Kp={Kp}, Ki={Ki}, Kd={Kd})',
                  fontsize=12, fontweight='bold')
    ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
    ax1.legend(); ax1.grid(True, alpha=0.3); ax1.axis('equal')

    ax2.plot(hcte, '-', color='#E74C3C', linewidth=1.5)
    ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax2.set_title('Cross-Track Error over Time', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Time step'); ax2.set_ylabel('CTE (m)')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("activity1_output.png", dpi=130, bbox_inches='tight')
    print("[VIZ] Saved -> activity1_output.png")
    plt.show()


def plot_comparison():
    """
    Run all three damping presets side by side -- makes lecture Slide 6
    (underdamped / overdamped / critically damped) concrete and tunable.
    """
    print("Running all three damping regimes for comparison ...")

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    colors = {'underdamped': '#E67E22',
              'overdamped': '#7F8C8D',
              'critically_damped': '#27AE60'}
    titles = {'underdamped': 'Underdamped (oscillating)',
              'overdamped': 'Overdamped (sluggish)',
              'critically_damped': 'Critically damped (ideal)'}

    for col, (name, gains) in enumerate(PRESETS.items()):
        path, hx, hy, hcte = run_simulation(**gains)
        c = colors[name]

        ax_traj = axes[0, col]
        ax_traj.plot(path[:, 0], path[:, 1], 'k--', linewidth=1.2, label='Reference')
        ax_traj.plot(hx, hy, '-', color=c, linewidth=2, label='Robot')
        ax_traj.set_title(f"{titles[name]}\nKp={gains['Kp']}, Kd={gains['Kd']}",
                          fontsize=11, fontweight='bold', color=c)
        ax_traj.set_xlabel('x (m)'); ax_traj.set_ylabel('y (m)')
        ax_traj.legend(fontsize=8); ax_traj.grid(True, alpha=0.3)
        ax_traj.set_ylim(-12, 12)

        ax_cte = axes[1, col]
        ax_cte.plot(hcte, '-', color=c, linewidth=1.3)
        ax_cte.axhline(0, color='gray', linewidth=0.8, linestyle='--')
        ax_cte.set_title('Cross-Track Error', fontsize=10)
        ax_cte.set_xlabel('Time step'); ax_cte.set_ylabel('CTE (m)')
        ax_cte.grid(True, alpha=0.3)
        ax_cte.set_ylim(-8, 8)

    plt.suptitle("PID Damping Regimes (lecture Slide 6)",
                 fontsize=13, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig("activity1_output.png", dpi=130, bbox_inches='tight')
    print("[VIZ] Saved -> activity1_output.png")
    plt.show()


if __name__ == '__main__':
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == '--compare':
        plot_comparison()
    else:
        plot_single(Kp, Ki, Kd)
        print()
        print("TIP: run  'python activity1_pid.py --compare'  to see all")
        print("     three damping regimes side by side (lecture Slide 6).")


---

## Activity 2: Activity2 Pure Pursuit

**File:** `activity2_pure_pursuit.py`

Run the code below to complete this activity.


In [ ]:
"""
Lab 6: Path Tracking
Activity 2: Pure Pursuit Controller
Author: Dr. Akram Fatayer

Description:
    A unicycle robot follows a sine-wave path using the geometric Pure Pursuit
    controller. Your task is to complete the look-ahead steering formula and
    explore how the look-ahead distance Ld affects tracking.

    Pure Pursuit (lecture Slides 7-8) has a SINGLE tuning knob -- the look-ahead
    distance Ld -- versus PID's three gains. This script also overlays the PID
    result from Activity 1 so you can compare the two controllers on the same
    path, exactly as the lecture's comparison table (Slide 10) describes.
"""

import numpy as np
import matplotlib.pyplot as plt

# --- ROBOT PARAMETERS ---
DT = 0.1          # Time step (seconds)
V  = 2.0          # Constant forward velocity (m/s)
L  = 2.5          # Wheelbase of the robot (meters)

# --- PURE PURSUIT PARAMETER (TUNE THIS) ---
# TODO: Tune the look-ahead distance and observe the trade-off (lecture Slide 8):
#   small Ld -> tight tracking but oscillation
#   large Ld -> smooth but cuts corners
Ld = 3.0          # Look-ahead distance (meters)

# Optional: adaptive look-ahead Ld = k * v  (lecture Slide 8).
# Set USE_ADAPTIVE = True to scale Ld with velocity. With constant V this just
# scales Ld, but it shows the mechanism used by Tesla Autopilot / ROS Nav Stack.
USE_ADAPTIVE = False
ADAPTIVE_K   = 1.5    # time constant (seconds); Ld = k * v


def generate_path():
    """Generates a sine wave path for the robot to follow."""
    x = np.linspace(0, 50, 500)
    y = 5 * np.sin(x / 5.0)
    return np.column_stack((x, y))


def get_lookahead_point(robot_pos, path, Ld):
    """
    Finds the first path point that is at least Ld away from the robot,
    searching forward from the nearest point.
    """
    distances   = np.linalg.norm(path - robot_pos, axis=1)
    nearest_idx = np.argmin(distances)

    target_idx = nearest_idx
    for i in range(nearest_idx, len(path)):
        if distances[i] >= Ld:
            target_idx = i
            break
    else:
        target_idx = len(path) - 1   # reached the end -> aim at last point

    return path[target_idx]


def pure_pursuit_control(robot_state, target_point, Ld, L):
    """
    Calculates the steering angle using the Pure Pursuit geometric model.

    Args:
        robot_state  : [x, y, theta]
        target_point : [x, y] look-ahead point
        Ld           : look-ahead distance
        L            : wheelbase

    Returns:
        steer : steering angle command (radians)
    """
    robot_x, robot_y, theta = robot_state
    target_x, target_y      = target_point

    # 1. Global angle to the target point
    alpha_global = np.arctan2(target_y - robot_y, target_x - robot_x)

    # 2. Angle to target RELATIVE to robot heading (normalized to [-pi, pi])
    alpha = np.arctan2(np.sin(alpha_global - theta),
                       np.cos(alpha_global - theta))

    # 3. Pure Pursuit steering formula (lecture Slide 7):
    #        delta = arctan( 2 * L * sin(alpha) / Ld )
    # TODO: implement this formula
    steer = np.arctan2(2.0 * L * np.sin(alpha), Ld)

    return steer


def run_simulation(Ld, steps=400, use_adaptive=False, adaptive_k=1.5):
    """Simulate Pure Pursuit tracking. Returns (path, hx, hy, hcte)."""
    path = generate_path()
    robot_state = np.array([0.0, 2.0, 0.0])

    history_x   = [robot_state[0]]
    history_y   = [robot_state[1]]
    history_cte = []

    for _ in range(steps):
        robot_pos = robot_state[:2]
        theta     = robot_state[2]

        # Adaptive look-ahead scales with speed (constant here, shown for concept)
        Ld_eff = adaptive_k * V if use_adaptive else Ld

        target_point = get_lookahead_point(robot_pos, path, Ld_eff)
        steer        = pure_pursuit_control(robot_state, target_point, Ld_eff, L)

        max_steer = np.radians(30)
        steer     = np.clip(steer, -max_steer, max_steer)

        # Track CTE for comparison metrics
        d = np.linalg.norm(path - robot_pos, axis=1)
        history_cte.append(d.min())

        # Unicycle kinematics
        robot_state[0] += V * np.cos(theta) * DT
        robot_state[1] += V * np.sin(theta) * DT
        robot_state[2] += (V / L) * np.tan(steer) * DT

        history_x.append(robot_state[0])
        history_y.append(robot_state[1])

        if robot_state[0] >= path[-1, 0]:
            break

    return path, history_x, history_y, history_cte


def try_import_pid():
    """Attempt to run the Activity 1 PID for an overlay comparison."""
    try:
        from activity1_pid import run_simulation as pid_sim, Kp, Ki, Kd
        path, hx, hy, hcte = pid_sim(Kp, Ki, Kd)
        return (hx, hy, hcte, f"PID (Kp={Kp}, Kd={Kd})")
    except Exception:
        return None


def main():
    print("Starting Pure Pursuit simulation...")
    mode = "adaptive (Ld = k*v)" if USE_ADAPTIVE else f"fixed Ld = {Ld} m"
    print(f"  Look-ahead mode: {mode}")

    path, hx, hy, hcte = run_simulation(Ld, use_adaptive=USE_ADAPTIVE,
                                        adaptive_k=ADAPTIVE_K)
    print("Simulation complete.")

    steady = [abs(c) for c in hcte[50:]] or hcte
    print(f"  Steady-state distance-to-path: max={max(steady):.3f} m, "
          f"mean={np.mean(steady):.3f} m")

    # Optional PID overlay
    pid = try_import_pid()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    ax1.plot(path[:, 0], path[:, 1], 'k--', linewidth=1.5, label='Reference path')
    ax1.plot(hx, hy, '-', color='#2980B9', linewidth=2,
             label=f'Pure Pursuit (Ld={Ld}m)')
    if pid:
        pid_hx, pid_hy, _, pid_label = pid
        ax1.plot(pid_hx, pid_hy, '-', color='#E67E22', linewidth=1.6,
                 alpha=0.8, label=pid_label)
    ax1.set_title('Pure Pursuit vs PID Path Following',
                  fontsize=12, fontweight='bold')
    ax1.set_xlabel('x (m)'); ax1.set_ylabel('y (m)')
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3); ax1.axis('equal')

    ax2.plot(hcte, '-', color='#2980B9', linewidth=1.5, label='Pure Pursuit')
    if pid:
        _, _, pid_hcte, _ = pid
        ax2.plot([abs(c) for c in pid_hcte], '-', color='#E67E22',
                 linewidth=1.4, alpha=0.8, label='PID')
    ax2.set_title('Distance to Path over Time', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Time step'); ax2.set_ylabel('|error| (m)')
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("activity2_output.png", dpi=130, bbox_inches='tight')
    print("[VIZ] Saved -> activity2_output.png")
    plt.show()

    print()
    print("REFLECTION QUESTIONS (answer in your lab report):")
    print("  Q1. Pure Pursuit has ONE tuning knob (Ld); PID has THREE.")
    print("      What does this trade off? When is each preferable?")
    print("  Q2. Re-run with Ld=1.0 then Ld=8.0. Describe the oscillation")
    print("      vs corner-cutting trade-off you observe (lecture Slide 8).")
    print("  Q3. Set USE_ADAPTIVE=True. Why does scaling Ld with speed help")
    print("      a real autonomous vehicle? (lecture Slide 8: Ld = k*v)")


if __name__ == '__main__':
    main()


---

## Activity 3: Activity3 Warehouse

**File:** `activity3_warehouse.py`

Run the code below to complete this activity.


In [ ]:
"""
Lab 6: Path Tracking
Activity 3: PyBullet Warehouse Navigation (Capstone Integration)
Author: Dr. Akram Fatayer

Description:
    The capstone of the autonomy stack. This script:
      1. Re-plans the warehouse route with the A* planner from Lab 5
         (imported directly -- no need to copy a path file between labs).
      2. Drives that route with YOUR Pure Pursuit controller from Activity 2.
      3. Shows the Husky robot autonomously following the planned path in 3-D.

    This closes the loop: Plan (Lab 5) -> Track (Lab 6).

PREREQUISITE:
    Place Lab 5's `activity1_astar.py` in this folder as `lab5_astar.py`
    (this package already includes it). If it is missing, the script falls
    back to a built-in A* so the lab still runs.
"""

import pybullet as p
import pybullet_data
import time
import math
import numpy as np
import os
import sys

# ==============================================================================
# A* PLANNER  -- imported from Lab 5
# ==============================================================================
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

try:
    from lab5_astar import a_star_search
    print("[PLANNER] Using A* imported from Lab 5 (lab5_astar.py).")
except Exception:
    # Minimal fallback so the lab runs even if the Lab 5 file is missing.
    import heapq

    def a_star_search(grid, start, goal):
        """Fallback 4-connected A* (same interface as Lab 5)."""
        def h(a, b):
            return abs(a[0] - b[0]) + abs(a[1] - b[1])

        open_set = [(0, start)]
        came_from = {}
        g_score = {start: 0}
        explored = set()
        rows, cols = grid.shape
        while open_set:
            _, cur = heapq.heappop(open_set)
            if cur in explored:
                continue
            explored.add(cur)
            if cur == goal:
                path = []
                while cur in came_from:
                    path.append(cur)
                    cur = came_from[cur]
                path.append(start)
                path.reverse()
                return path, explored
            for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
                nb = (cur[0] + dr, cur[1] + dc)
                if 0 <= nb[0] < rows and 0 <= nb[1] < cols and grid[nb[0], nb[1]] == 0:
                    tg = g_score[cur] + 1
                    if nb not in g_score or tg < g_score[nb]:
                        g_score[nb] = tg
                        came_from[nb] = cur
                        heapq.heappush(open_set, (tg + h(nb, goal), nb))
        return None, explored

    print("[PLANNER] lab5_astar.py not found -- using built-in fallback A*.")


# ==============================================================================
# ROBOT PARAMETERS  (Clearpath Husky -- realistic values)
# ==============================================================================
WHEEL_RADIUS = 0.165      # Husky wheel radius (m)
WHEEL_TRACK  = 0.555      # distance between left and right wheels (m) -- the "L"
                          # in the differential-drive conversion
MAX_WHEEL_W  = 12.0       # max wheel angular velocity (rad/s)

# Pure Pursuit tuning for the warehouse scale
TARGET_V = 1.2            # forward speed command (m/s)
Ld       = 1.0            # look-ahead distance (m)

# ==============================================================================
# GRID  <->  WORLD  (must match Lab 5 Activity 3)
# ==============================================================================
CELL_SIZE = 1.0
GRID_ROWS = 12
GRID_COLS = 12


def cell_to_world(row, col):
    return col * CELL_SIZE + CELL_SIZE / 2, row * CELL_SIZE + CELL_SIZE / 2


def build_warehouse_grid():
    """Same warehouse layout as Lab 5 Activity 3 (0 = free, 1 = shelf)."""
    grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=int)
    grid[3:7, 2]   = 1
    grid[3,   2:6] = 1
    grid[3:7, 8]   = 1
    grid[3,   7:9] = 1
    grid[8,   3:9] = 1
    return grid


# ==============================================================================
# PURE PURSUIT  (your Activity 2 controller)
# ==============================================================================

def get_lookahead_point(robot_pos, path, Ld):
    """First path point at least Ld away, searching forward from nearest."""
    distances   = np.linalg.norm(path - robot_pos, axis=1)
    nearest_idx = np.argmin(distances)
    for i in range(nearest_idx, len(path)):
        if distances[i] >= Ld:
            return path[i]
    return path[-1]


def pure_pursuit_control(robot_state, target_point, Ld, L):
    """
    Pure Pursuit steering (lecture Slide 7):
        delta = arctan( 2 * L * sin(alpha) / Ld )
    """
    robot_x, robot_y, theta = robot_state
    target_x, target_y      = target_point

    alpha_global = np.arctan2(target_y - robot_y, target_x - robot_x)
    alpha        = np.arctan2(np.sin(alpha_global - theta),
                              np.cos(alpha_global - theta))
    steer        = np.arctan2(2.0 * L * np.sin(alpha), Ld)
    return steer


# ==============================================================================
# LAYER 3 -- CONTROLLER: unicycle (v, omega) -> differential wheel speeds
# ==============================================================================

def steering_to_wheel_velocities(v, steer, L, wheel_radius):
    """
    Convert a unicycle command into left/right wheel ANGULAR velocities.

    From the lecture (Slide 4):
        omega = v / L * tan(delta)             (bicycle/unicycle steering)
        v_R   = v + omega * L / 2              (right wheel linear speed)
        v_L   = v - omega * L / 2              (left  wheel linear speed)
        w     = v_wheel / wheel_radius          (linear -> angular)

    Args:
        v            : forward linear velocity (m/s)
        steer        : steering angle delta (rad)
        L            : wheel track (m)
        wheel_radius : wheel radius (m)

    Returns:
        wL, wR : left and right wheel angular velocities (rad/s)
    """
    omega = (v / L) * math.tan(steer)     # body angular velocity (rad/s)

    v_left  = v - omega * L / 2.0
    v_right = v + omega * L / 2.0

    wL = v_left  / wheel_radius
    wR = v_right / wheel_radius

    wL = float(np.clip(wL, -MAX_WHEEL_W, MAX_WHEEL_W))
    wR = float(np.clip(wR, -MAX_WHEEL_W, MAX_WHEEL_W))
    return wL, wR


def apply_wheel_velocities(robot_id, wL, wR):
    """Send wheel angular velocities to the Husky joints."""
    # Husky joint indices: 2=front_left, 3=front_right, 4=rear_left, 5=rear_right
    left_wheels  = [2, 4]
    right_wheels = [3, 5]
    for j in left_wheels:
        p.setJointMotorControl2(robot_id, j, p.VELOCITY_CONTROL,
                                targetVelocity=wL, force=40)
    for j in right_wheels:
        p.setJointMotorControl2(robot_id, j, p.VELOCITY_CONTROL,
                                targetVelocity=wR, force=40)


# ==============================================================================
# PYBULLET SCENE
# ==============================================================================

def get_robot_state(robot_id):
    pos, orn = p.getBasePositionAndOrientation(robot_id)
    yaw      = p.getEulerFromQuaternion(orn)[2]
    return np.array([pos[0], pos[1], yaw])


def setup_warehouse(grid, start_cell):
    p.connect(p.DIRECT if IN_COLAB else p.GUI)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)

    p.loadURDF("plane.urdf")

    # Shelf obstacles from the grid
    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            if grid[r, c] == 1:
                x, y = cell_to_world(r, c)
                col = p.createCollisionShape(p.GEOM_BOX,
                                             halfExtents=[0.45, 0.45, 0.6])
                vis = p.createVisualShape(p.GEOM_BOX, halfExtents=[0.45, 0.45, 0.6],
                                          rgbaColor=[0.55, 0.38, 0.18, 1])
                p.createMultiBody(baseMass=0, baseCollisionShapeIndex=col,
                                  baseVisualShapeIndex=vis, basePosition=[x, y, 0.6])

    # Robot at the start cell
    sx, sy = cell_to_world(*start_cell)
    robot_id = p.loadURDF("husky/husky.urdf", [sx, sy, 0.15],
                          p.getQuaternionFromEuler([0, 0, 0]))
    for _ in range(60):
        p.stepSimulation()

    p.resetDebugVisualizerCamera(cameraDistance=14, cameraYaw=30,
                                 cameraPitch=-55, cameraTargetPosition=[6, 6, 0])
    return robot_id


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    print("=" * 60)
    print("  Lab 6 Activity 3 -- Capstone: Plan (A*) + Track (Pure Pursuit)")
    print("=" * 60)

    # --- Plan with A* (imported from Lab 5) ---
    grid       = build_warehouse_grid()
    start_cell = (1, 1)
    goal_cell  = (10, 10)

    print("[PLAN] Running A* on the warehouse grid ...")
    path_cells, explored = a_star_search(grid, start_cell, goal_cell)
    if path_cells is None:
        print("[PLAN] ERROR: A* found no path. Aborting.")
        return

    # Convert grid cells to world-coordinate waypoints
    path = np.array([cell_to_world(r, c) for (r, c) in path_cells])
    print(f"[PLAN] Path found: {len(path)} waypoints, {len(explored)} cells explored.")

    # --- PyBullet ---
    print("[SIM] Launching PyBullet warehouse ...")
    robot_id = setup_warehouse(grid, start_cell)

    # Draw the A* path as a green line
    for i in range(len(path) - 1):
        p.addUserDebugLine([path[i][0], path[i][1], 0.15],
                           [path[i + 1][0], path[i + 1][1], 0.15],
                           lineColorRGB=[0.1, 0.85, 0.25], lineWidth=3)

    # Mark the goal
    goal_x, goal_y = cell_to_world(*goal_cell)
    gv = p.createVisualShape(p.GEOM_SPHERE, radius=0.3, rgbaColor=[0.1, 0.5, 1, 0.9])
    p.createMultiBody(baseMass=0, baseVisualShapeIndex=gv,
                      basePosition=[goal_x, goal_y, 0.3])

    print("[TRACK] Driving the path with Pure Pursuit ...")
    print(f"        Ld={Ld} m, target speed={TARGET_V} m/s, "
          f"wheel track L={WHEEL_TRACK} m")

    reached = False
    for step in range(6000):
        state     = get_robot_state(robot_id)
        robot_pos = state[:2]

        # Pure Pursuit -> steering angle
        target = get_lookahead_point(robot_pos, path, Ld)
        steer  = pure_pursuit_control(state, target, Ld, WHEEL_TRACK)
        steer  = float(np.clip(steer, -np.radians(35), np.radians(35)))

        # Visualize the current look-ahead target
        p.addUserDebugLine([robot_pos[0], robot_pos[1], 0.25],
                           [target[0], target[1], 0.25],
                           lineColorRGB=[1, 0.2, 0.2], lineWidth=2, lifeTime=0.05)

        # Convert to wheel speeds and apply (Layer 3)
        wL, wR = steering_to_wheel_velocities(TARGET_V, steer,
                                              WHEEL_TRACK, WHEEL_RADIUS)
        apply_wheel_velocities(robot_id, wL, wR)

        # Goal check
        if np.linalg.norm(path[-1] - robot_pos) < 0.6:
            print(f"[TRACK] Goal reached in {step} steps!")
            apply_wheel_velocities(robot_id, 0, 0)
            reached = True
            break

        p.stepSimulation()
        time.sleep(1.0 / 240.0)

    if not reached:
        print("[TRACK] Simulation ended before reaching the goal "
              "(try increasing Ld or step count).")

    print()
    print("REFLECTION QUESTIONS (answer in your lab report):")
    print("  Q1. The A* path is jagged (grid waypoints). How does the")
    print("      look-ahead distance Ld smooth the robot's actual motion?")
    print("  Q2. This integrates Lab 5 (Plan) with Lab 6 (Track). Which")
    print("      part is the 'global planner' and which is the 'local'?")
    print("  Q3. The robot cuts the corners of the A* path slightly. Is")
    print("      that a bug or expected Pure Pursuit behavior? Explain.")
    print()
    print("[SIM] Close the window to exit.")

    try:
        while p.isConnected():
            p.stepSimulation()
            time.sleep(1.0 / 240.0)
    except Exception:
        pass
    p.disconnect()


if __name__ == '__main__':
    main()


---

## Summary & Next Steps

Congratulations on completing this lab! Before moving on:

1. **Review** your outputs and make sure they match expected behavior
2. **Experiment** by modifying parameters and observing the effects
3. **Reflect** on what you learned — write a brief paragraph in your report

---

*Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/) · Al-Zaytoonah University of Jordan*
